In [ ]:
import os
import pandas as pd
import re

In [ ]:
# Set the base directory
base_dir = "/project3/data/results/addeam-orf-clustering"  # Change this to your actual directory

In [ ]:
# List of cluster numbers to check in k3 directory
clusters_to_check = [1, 2, 3]

# Dictionary to store results
valid_clusters = {}

# Function to extract substring after 'config'
def extract_substring(sample_name):
    match = re.search(r"config[^_]*_(.*)", sample_name)  # Look for 'config' followed by an underscore
    return match.group(1) if match else sample_name  # Return extracted part, or original if no match

# Iterate over all subdirectories
for subdir in os.listdir(base_dir):
    
    # if not "101." in subdir:
    #     continue
    
    subdir_path = os.path.join(base_dir, subdir)
    
    # Ensure it's a directory and ends with "clusters"
    if not os.path.isdir(subdir_path) or not subdir.endswith("clustersMin100"):
        continue

    # Dictionary to store clusters that fulfill the criteria for this subdir
    matching_clusters = {}

    # Check each cluster file in GMM/k3/
    for cluster in clusters_to_check:
        profile_5p = os.path.join(subdir_path, "GMM", "k3", f"cluster_{cluster}_weighted_profile_5p.prof")
        profile_3p = os.path.join(subdir_path, "GMM", "k3", f"cluster_{cluster}_weighted_profile_3p.prof")

        # Check if the files exist
        if not os.path.exists(profile_5p) or not os.path.exists(profile_3p):
            continue

        # Read the file, skipping the first row (header)
        try:
            df = pd.read_csv(profile_5p, sep="\t", skiprows=1, header=None)

            # Check if any value exceeds 0.05
            if (df > 0.05).any().any():
                # Initialize a list to store samples and paths
                matching_clusters[cluster] = {
                    "samples": [],
                    "profile_5p": profile_5p,
                    "profile_3p": profile_3p,
                }

        except Exception as e:
            print(f"Error reading {profile_5p}: {e}")

    # If no clusters met the criteria, continue to the next subdir
    if not matching_clusters:
        continue

    # Load the cluster report file
    report_file = os.path.join(subdir_path, "GMM", "k3", "cluster_report_k3.tsv")
    if os.path.exists(report_file):
        try:
            # Read the cluster report TSV
            report_df = pd.read_csv(report_file, sep="\t")

            # Iterate through rows and assign each sample to the cluster with the highest probability
            for _, row in report_df.iterrows():
                # Find the cluster with the highest probability
                best_cluster = max(clusters_to_check, key=lambda c: row[f"Cluster{c}"])

                # If this cluster is in our valid list, extract the substring and store it
                if best_cluster in matching_clusters:
                    clean_sample_name = extract_substring(row["SampleName"])
                    matching_clusters[best_cluster]["samples"].append(clean_sample_name)

        except Exception as e:
            print(f"Error reading {report_file}: {e}")

    # Store results in the main dictionary
    valid_clusters[subdir] = matching_clusters



In [ ]:
# **SAVE RESULTS TO A FILE**

output_file = "/project3/data/results/other-evals/ancientness/ancient_orfs_min100_only_headers.tsv"
with open(output_file, "w") as f:   
    for subdir, clusters in valid_clusters.items():
        for cluster, data in clusters.items():
            for sample in data["samples"]:
                pureSample = sample.rsplit("_",1)[0]
                f.write(f"{pureSample}\n")

print(f"Results saved to {output_file}")
